In [ ]:
# 保存网页内容

import requests
from bs4 import BeautifulSoup
import os

# 目标URL
url = "https://www.gushiji.cc/gushi/p1.html"

# 设置保存路径
save_dir = r"G:\Apps\VSCode\Projects\Learn and Think\Call API\Data"
save_filename = "gushi_page1.html"
save_path = os.path.join(save_dir, save_filename)

# 发送HTTP请求
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0"}
response = requests.get(url, headers=headers)

# 检查请求是否成功
if response.status_code == 200:
    # 获取网页内容
    content = response.text
    # 保存到文件
    with open(save_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"网页内容已保存到：{save_path}")
else:
    print(f"请求失败，状态码：{response.status_code}")

网页内容已保存到：G:\Apps\VSCode\Projects\Learn and Think\Call API\Data\gushi_page1.html


https://www.gushiji.cc/gushi/p1.html

ul class="simple-shiciqu has-author main-data"



In [7]:
<li>02.《<a href="/gushi/b2101.html">国风·周南·葛覃</a>》<a href="/shiren/xianqin.html">先秦</a><span>·</span><a class="author" href="/shiren/shijing/">诗经</a> <span class="content">葛之覃兮，施于中谷，维叶萋萋。黄鸟于飞，集于灌木，其鸣喈喈。葛之覃兮，施于中谷，维叶莫莫。是刈是濩，为絺为綌，服之无斁。言告师氏，言告言归。薄污我私，薄浣我衣。害浣害否？归宁父母。</span></li>

SyntaxError: invalid character '《' (U+300A) (2155137581.py, line 1)

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import os


def get_poem_content(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    poems = {}

    # 获取诗词列表
    # poem_list = soup.find("ul", class_="simple-shiciqu has-author main-data")
    poem_list = soup.find("ul", attrs={"class": "simple-shiciqu has-author main-data"})
    if not poem_list:
        return poems

    # 遍历每个诗词条目
    for item in poem_list.find_all("li"):
        # 提取标题
        title_tag = item.find_all("a")[0]
        title = title_tag.text if title_tag else ""

        # 提取朝代
        dynasty_tag = item.find_all("a")[1]
        dynasty = dynasty_tag.text if dynasty_tag else ""

        # 提取作者
        author_tag = item.find("a", class_="author")
        author = author_tag.text if author_tag else ""

        # 提取内容
        content_tag = item.find("span", class_="content")
        content = content_tag.text.strip() if content_tag else ""

        if title and content:
            poems[title] = {"content": content, "author": author, "dynasty": dynasty}

    return poems


def save_to_json(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)


# 使用示例
if __name__ == "__main__":
    url = "https://www.gushiji.cc/gushi/p1.html"
    poems = get_poem_content(url)
    save_dir = r"G:\Apps\VSCode\Projects\Learn and Think\Call API\Data"
    save_filename = "at1_zh.json"
    save_path = os.path.join(save_dir, save_filename)
    save_to_json(poems, save_path)

In [4]:
import requests
from bs4 import BeautifulSoup
import os
import json


def save_to_json(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)


def crawl_all_pages(base_url, save_dir, filename="at_zh.json"):
    """持续爬取所有页面直到遇到空页面，并将所有内容保存到同一个文件中"""
    all_poems = {}
    page = 1

    while True:
        url = f"{base_url}p{page}.html"
        print(f"\r正在爬取第 {page} 页...", end="")

        try:
            response = requests.get(url)
            response.encoding = "utf-8"  # 确保响应使用UTF-8编码
            if response.status_code != 200:
                print(f"请求失败，状态码：{response.status_code}")
                break

            soup = BeautifulSoup(response.text, "html.parser")
            poem_list = soup.find("ul", attrs={"class": "simple-shiciqu has-author main-data"})

            if not poem_list or not poem_list.find_all("li"):
                print(f"第 {page} 页无内容，爬取结束")
                break

            page_poems = {}
            for item in poem_list.find_all("li"):
                # 提取标题
                title_tag = item.find_all("a")[0]
                title = title_tag.text if title_tag else ""

                # 提取朝代
                dynasty_tag = item.find_all("a")[1]
                dynasty = dynasty_tag.text if dynasty_tag else ""

                # 提取作者
                author_tag = item.find("a", class_="author")
                author = author_tag.text if author_tag else ""

                # 提取内容
                content_tag = item.find("span", class_="content")
                content = content_tag.text.strip() if content_tag else ""

                if "/" in title:
                    titles = title.split("/")
                    title = max(titles, key=len).strip()

                if title and content:
                    page_poems[f"{title} - {author}"] = {"content": content, "author": author, "dynasty": dynasty}

            if page_poems:
                all_poems.update(page_poems)
                # 每爬取一页就保存一次
                save_path = os.path.join(save_dir, filename)
                save_to_json(all_poems, save_path)
                print(f"\r                             第 {page} 页数据已保存，当前总诗词数：{len(all_poems)}", end="")
                page += 1
            else:
                print(f"第 {page} 页无有效数据，爬取结束")
                break

        except Exception as e:
            print(f"爬取第 {page} 页时发生错误：{str(e)}")
            break


if __name__ == "__main__":
    base_url = "https://www.gushiji.cc/tangshisanbaishou/"
    save_dir = r"G:\Apps\VSCode\Projects\Learn and Think\Call API\Data"
    crawl_all_pages(base_url, save_dir)

正在爬取第 7 页...                 第 6 页数据已保存，当前总诗词数：316第 7 页无内容，爬取结束
